# Exercise 2: ACLED Notes

Here the goal is not to build a complicated model. The goal is to show that the `notes` field can help characterize protests and riots beyond their event labels.


## Two contrasting country cases

We use two ready-made ACLED extracts:

- **Spain**: mostly demonstrations and social protest
- **Sudan**: protests and riots in a much more conflict-intense environment

This contrast is useful because the same event labels can hide very different dynamics.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

candidates = [Path.cwd(), Path.cwd() / 'sessions' / 'in_class_assignment']
assignment_root = next(path for path in candidates if (path / 'data').exists())
data_dir = assignment_root / 'data' / 'acled_notes'

spain = pd.read_parquet(data_dir / 'acled_spain.parquet')
sudan = pd.read_parquet(data_dir / 'acled_sudan.parquet')

for df in (spain, sudan):
    df['event_date'] = pd.to_datetime(df['event_date'])

print('Spain:', spain.shape)
print('Sudan:', sudan.shape)


In [ ]:
def monthly_protest_riot_counts(df: pd.DataFrame) -> pd.DataFrame:
    subset = df.loc[df['event_type'].isin(['Protests', 'Riots'])].copy()
    subset['month'] = subset['event_date'].dt.to_period('M').dt.to_timestamp()
    counts = (
        subset.groupby(['month', 'event_type'])
        .size()
        .unstack(fill_value=0)
        .sort_index()
    )
    for col in ['Protests', 'Riots']:
        if col not in counts.columns:
            counts[col] = 0
    counts['total'] = counts['Protests'] + counts['Riots']
    counts['total_ma3'] = counts['total'].rolling(3, min_periods=1).mean()
    return counts.reset_index()

spain_monthly = monthly_protest_riot_counts(spain)
sudan_monthly = monthly_protest_riot_counts(sudan)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=False)

for ax, df_plot, title in [
    (axes[0], spain_monthly, 'Spain: protests and riots'),
    (axes[1], sudan_monthly, 'Sudan: protests and riots'),
]:
    ax.plot(df_plot['month'], df_plot['Protests'], label='Protests', color='#2f6db3', linewidth=2)
    ax.plot(df_plot['month'], df_plot['Riots'], label='Riots', color='#b22222', linewidth=2)
    ax.plot(df_plot['month'], df_plot['total_ma3'], label='Total (3M average)', color='black', linestyle='--', linewidth=1.8)
    ax.set_title(title)
    ax.set_ylabel('Monthly incidence')
    ax.grid(alpha=0.3)
    ax.legend(loc='upper left')

axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
note_examples = pd.concat(
    [
        spain.loc[spain['event_type'].isin(['Protests', 'Riots']), ['event_date', 'event_type', 'notes']].head(2).assign(country='Spain'),
        sudan.loc[sudan['event_type'].isin(['Protests', 'Riots']), ['event_date', 'event_type', 'notes']].head(2).assign(country='Sudan'),
    ],
    ignore_index=True,
)
print(note_examples[['country', 'event_type', 'notes']].to_string(index=False))


In [ ]:
dictionaries = {
    'economic_grievance': ['wage', 'salary', 'cost of living', 'housing', 'prices', 'bonus', 'pension', 'working conditions'],
    'institutional_claims': ['government', 'legal', 'approval process', 'rights', 'repression', 'justice', 'election'],
    'security_escalation': ['police', 'scuffles', 'thrown', 'clashes', 'artillery', 'shelling', 'killed', 'attacked'],
}

def build_note_indices(df: pd.DataFrame, dictionaries: dict[str, list[str]]) -> pd.DataFrame:
    subset = df.loc[df['event_type'].isin(['Protests', 'Riots'])].copy()
    subset['month'] = subset['event_date'].dt.to_period('M').dt.to_timestamp()
    text = subset['notes'].fillna('').str.lower()
    for name, terms in dictionaries.items():
        subset[name] = text.apply(lambda value: sum(term in value for term in terms))
    monthly = subset.groupby('month')[list(dictionaries)].mean().reset_index()
    return monthly

spain_indices = build_note_indices(spain, dictionaries)
sudan_indices = build_note_indices(sudan, dictionaries)
spain_indices.head()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=False)

for ax, df_plot, title in [
    (axes[0], spain_indices, 'Spain: note-based protest characterization'),
    (axes[1], sudan_indices, 'Sudan: note-based protest characterization'),
]:
    for col, color in [
        ('economic_grievance', '#2f6db3'),
        ('institutional_claims', '#d95f02'),
        ('security_escalation', '#7a1fa2'),
    ]:
        ax.plot(df_plot['month'], df_plot[col], label=col.replace('_', ' ').title(), linewidth=2, color=color)
    ax.set_ylabel('Average keyword hits per event')
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend(loc='upper left')

axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


## Ideas

You do not need to keep these exact dictionaries. Better ideas are welcome.

Possible directions:

- Characterize **violent events** from **peaceful events**, decompose the incidence of protests and riots.
- Distinguish **economic protests** from **institutional protests**.
- Build a monthly **repression / escalation** index from the notes.
- Ask whether `Riots` are just more intense `Protests`, or whether the notes suggest a different composition.
- Compare how much richer the notes are in Spain versus Sudan for the same broad event labels.
